In [1]:
%reload_ext autoreload
%autoreload 2

import sys
import os
import sklearn
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm

from stnd.utility.data_utils import make_or_load_from_cache
from stnd.utility.utils import apply_random_seed
import sys
import os
import sklearn
import numpy as np
import pandas as pd
from datasets import load_dataset
import json


FIRST_RUN = False
ROOT_PATH = os.path.join(os.path.dirname(os.path.abspath("")))
sys.path.insert(0, ROOT_PATH)
from experiments import (
    RANDOM_SEED,
    make_train_test_model_embeddings,
    make_cache_subpath,
    make_disagreement_scores_dict,
    make_fitted_weights
)
from utils import (
    lb_scenarios,
    dump_pickle,
    load_pickle,
    prepare_and_split_data
)
from plots import (
    MODEL_OUTPUTS_PATH,
    load_scores,
    safe_spearmanr
)
from selection import sample_items
from run_experiment import load_and_split_model_outputs
from acc import (
    compute_true_acc,
    compute_acc_knn
)
from scripts.download_leaderboard import MMLU_SUBSCENARIOS
# from utils_for_notebooks import read_per_model_info
from utils_for_notebooks import pad_predictions
from scripts.evaluate_mmlu import evaluate_mmlu
from scripts.extract_model_outputs_from_raw_data_v2 import MMLU_PRO_SCENARIO_SUFFIX
sys.path.pop(0);


CACHE_DIR = "./cache_dir"


/home/oh/arubinstein17/github/disco-public/envs/disco_env_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## functions

In [2]:
def _get_scenario_key(entry):
    """Return the key that ends with __leaderboard_mmlu_pro, or None."""
    if not isinstance(entry, dict):
        return None
    for k in entry:
        if k.endswith(MMLU_PRO_SCENARIO_SUFFIX):
            return k
    return None


def extract_source_outputs_from_v2_raw(raw_path, pad_to_size=31):
    """
    Load v2 leaderboard raw pickle and build source_outputs dict.

    Raw pickle format: {model_name: {scenario_name: {correctness, resps, ...}}}

    pad_to_size: size of the last dimension of predictions (pad with -inf).
    """
    raw = load_pickle(raw_path)

    # models = []
    scenario_key = None
    n_questions = None

    models_dict = {}

    for model_name, scenarios_dict in tqdm(raw.items()):
        if not isinstance(scenarios_dict, dict):
            continue
        sk = _get_scenario_key(scenarios_dict)
        if sk is None:
            continue
        rec = scenarios_dict[sk]
        if rec is None:
            continue
        correctness = rec.get("correctness")
        if correctness is None:
            continue
        try:
            nc = len(correctness)
        except TypeError:
            continue
        if scenario_key is None:
            scenario_key = sk
            n_questions = nc
        # elif sk != scenario_key or nc != n_questions:
        elif nc != n_questions:
            continue
        # models.append((model_name, rec))
        models_dict[model_name] = rec

    # if not models or scenario_key is None or n_questions is None:
    #     raise ValueError(
    #         "No valid model data found in raw pickle; need at least one model "
    #         "with non-None correctness for the MMLU-Pro scenario."
    #     )

    # n_models = len(models)
    # n_answers = 1

    # correctness_arr = np.zeros((n_models, n_questions, 1), dtype=np.float64)
    # predictions_arr = np.full(
    #     (n_models, n_questions, pad_to_size), -np.inf, dtype=np.float64
    # )

    return models_dict

    # for i, (model_name, rec) in enumerate(models):
    #     corr = rec["correctness"]
    #     if hasattr(corr, "__iter__") and not isinstance(corr, (str, bytes)):
    #         corr = list(corr)
    #     else:
    #         corr = [float(corr)] * n_questions
    #     correctness_arr[i, :, 0] = np.array(
    #         corr[:n_questions], dtype=np.float64
    #     )

    #     resps = rec.get("resps") or rec.get("filtered_resps")
    #     pred = _resps_to_prediction_indices(resps, n_questions, max_answers=1)
    #     predictions_arr[i, :, :1] = pred[:, :1]

    # # Same format as two_stages.build_outputs_dict / source_outputs.pkl
    # scenarios_map = {scenario_key: list(range(n_questions))}
    # models_map = {name: i for i, (name, _) in enumerate(models)}
    # datapoints_map = {i: i for i in range(n_questions)}

    # return {
    #     "predictions": predictions_arr,
    #     "correctness": correctness_arr,
    #     "Models": models_map,
    #     "Datapoints": datapoints_map,
    #     "Scenarios": scenarios_map,
    # }

## check remote responses

### Save single model

In [4]:
# model_name = 'open-llm-leaderboard/JungZoona__T3Q-qwen2.5-14b-v1.0-e3-details'
model_name = "open-llm-leaderboard/01-ai__Yi-9B-details"

In [16]:
if FIRST_RUN:
    raw_path = "/home/oh/arubinstein17/github/disco-public/data/leaderboard_mmlu_pro_raw_all.pickle"
    # raw_path = "/home/oh/arubinstein17/github/disco-public/data/leaderboard_mmlu_pro_raw.pickle"
    models_dict = extract_source_outputs_from_v2_raw(raw_path)
# 12 min loading

100%|██████████| 6505/6505 [00:00<00:00, 874457.47it/s]


In [5]:
import pickle

# model_name = 'open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details'

if FIRST_RUN:
    with open(model_name.replace("/", "_") + ".pkl", "wb") as f:
        pickle.dump(models_dict[model_name], f)

In [6]:
with open(model_name.replace("/", "_") + ".pkl", "rb") as f:
    single_model_dict = pickle.load(f)

In [9]:
print(single_model_dict.keys())

dict_keys(['dates', 'doc', 'resps', 'filtered_resps', 'target', 'arguments', 'correctness'])


In [8]:
single_model_dict['resps']

Column([[[['-3.1902904510498047', 'False']], [['-3.4402904510498047', 'False']], [['-3.8152904510498047', 'False']], [['-2.6902904510498047', 'False']], [['-1.0652905702590942', 'True']], [['-2.1902904510498047', 'False']], [['-3.9402904510498047', 'False']], [['-4.690290451049805', 'False']], [['-1.1902905702590942', 'False']]], [[['-4.589632034301758', 'False']], [['-5.339632034301758', 'False']], [['-5.964632034301758', 'False']], [['-5.964632034301758', 'False']], [['-0.08963188529014587', 'True']], [['-7.339632034301758', 'False']], [['-7.214632034301758', 'False']], [['-8.089632034301758', 'False']], [['-7.839632034301758', 'False']], [['-5.839632034301758', 'False']]], [[['-1.7078464031219482', 'False']], [['-2.0828464031219482', 'False']], [['-2.3328464031219482', 'False']], [['-1.5828464031219482', 'True']], [['-1.9578464031219482', 'False']], [['-2.9578464031219482', 'False']], [['-3.2078464031219482', 'False']], [['-4.082846641540527', 'False']], [['-3.7078464031219482', 'Fa

In [14]:
import pandas as pd
import random

csv_path = "/home/oh/arubinstein17/github/disco-public/benchmark_csvs/open-llm-leaderboard-v2.csv"
df = pd.read_csv(csv_path)

filtered_df = df[df['Type'].str.contains("pretrained", na=False)]
filtered_df = df[df['fullname'].str.contains("/", na=False)]
SELECTED_MODELS = random.sample(list(filtered_df['fullname']), 10)
print(SELECTED_MODELS)

['Deci/DeciLM-7B-instruct', 'Sakalti/SJT-8B-V1.1', 'T145/ZEUS-8B-V26', 'marcuscedricridia/Hush-Qwen2.5-7B-v1.2', 'godlikehhd/alpaca_data_score_max_0.7_2600', '3rd-Degree-Burn/L-3.1-Science-Writer-8B', 'ClaudioItaly/Book-Gut12B', 'jaspionjader/bh-19', 'Lawnakk/BBALAW1.61', 'fblgit/miniclaus-qw1.5B-UNAMGS']


In [19]:
if FIRST_RUN:
    dict_to_save = {}
    for model_name in SELECTED_MODELS:
        dict_key = "open-llm-leaderboard/" + model_name.replace("/", "__") + "-details"
        dict_to_save[model_name] = models_dict[dict_key]
    with open("selected_models.pkl", "wb") as f:
        pickle.dump(dict_to_save, f)

In [20]:
with open("selected_models.pkl", "rb") as f:
    candidate_models = pickle.load(f)

In [21]:
print(candidate_models.keys())

dict_keys(['Deci/DeciLM-7B-instruct', 'Sakalti/SJT-8B-V1.1', 'T145/ZEUS-8B-V26', 'marcuscedricridia/Hush-Qwen2.5-7B-v1.2', 'godlikehhd/alpaca_data_score_max_0.7_2600', '3rd-Degree-Burn/L-3.1-Science-Writer-8B', 'ClaudioItaly/Book-Gut12B', 'jaspionjader/bh-19', 'Lawnakk/BBALAW1.61', 'fblgit/miniclaus-qw1.5B-UNAMGS'])


### Debug

In [7]:

model_name_lucie = "open-llm-leaderboard/OpenLLM-France__Lucie-7B-details"

In [18]:
import pickle

# model_name = 'open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details'

if FIRST_RUN:
    with open(model_name.replace("/", "_") + ".pkl", "wb") as f:
        pickle.dump(models_dict[model_name], f)

In [8]:
with open(model_name_lucie.replace("/", "_") + ".pkl", "rb") as f:
    single_model_dict_lucie = pickle.load(f)

In [20]:
# raw_path = "/home/oh/arubinstein17/github/disco-public/data/leaderboard_mmlu_pro_raw_all.pickle"
raw_path = "/home/oh/arubinstein17/github/disco-public/data/leaderboard_mmlu_pro_raw.pickle"
models_dict = extract_source_outputs_from_v2_raw(raw_path)

100%|██████████| 4/4 [00:00<00:00, 73908.44it/s]


In [21]:
models_dict.keys()

dict_keys(['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details', 'open-llm-leaderboard/LeroyDyer__SpydazWeb_AI_HumanAI_011_INSTRUCT-details', 'open-llm-leaderboard/Youlln__ECE-MIRAGE-1-12B-details', 'open-llm-leaderboard/jaspionjader__bh-39-details'])

In [ ]:
models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']

In [23]:
models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details'].keys()

dict_keys(['dates', 'doc', 'resps', 'filtered_resps', 'target', 'arguments', 'correctness'])

In [24]:
import pickle

with open("matter_0.2-7B-DPO-details.pkl", "wb") as f:
    pickle.dump(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details'], f)



In [25]:
with open("matter_0.2-7B-DPO-details.pkl", "rb") as f:
    single_model_dict = pickle.load(f)

In [26]:
print(len(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']["arguments"]))
print(len(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']["filtered_resps"]))
print(len(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']["resps"]))
print(len(models_dict['open-llm-leaderboard/0-hero__Matter-0.2-7B-DPO-details']["correctness"]))




12032
12032
12032
12032


## Check local responses

In [12]:
local_path = "/home/oh/arubinstein17/github/lm-eval-fork/output/mmlu_pro/llmeval135/01-ai__Yi-9B/samples_leaderboard_mmlu_pro_2026-02-14T11-47-10.579194.jsonl"



In [ ]:
# Extract mmlu_pro_prompts_examples from gen_args_0.arg_0 and save in mmlu_prompts_examples.json format
import string

def doc_to_query(doc):
    """Build query string from doc (question + options + Answer:)."""
    s = f"{doc['question']}\n"
    for i in range(len(doc["options"])):
        s += f"{string.ascii_uppercase[i]}. {doc['options'][i]}\n"
    s += "Answer:"
    return s

def extract_mmlu_pro_prompts_examples(jsonl_path, output_path):
    """Extract prompts in mmlu_prompts_examples format from lm-eval JSONL samples."""
    examples = []
    with open(jsonl_path) as f:
        for line in f:
            rec = json.loads(line)
            doc = rec["doc"]
            arg_0 = rec["arguments"]["gen_args_0"]["arg_0"]  # full_prompt
            query = doc_to_query(doc)
            choices = [string.ascii_uppercase[i] for i in range(len(doc["options"]))]
            gold = doc["answer_index"]
            example_str = query + " " + doc["answer"]
            examples.append({
                "example": example_str,
                "full_prompt": arg_0,
                "query": query,
                "choices": choices,
                "gold": gold,
            })
    with open(output_path, "w") as out:
        json.dump(examples, out, indent=2)
    print(f"Saved {len(examples)} records to {output_path}")
    return examples

out_path = os.path.join(ROOT_PATH, "notebooks", "mmlu_pro_prompts_examples.json")
extract_mmlu_pro_prompts_examples(local_path, out_path)

## Compare local vs remote questions

In [ ]:
# single_model_dict = single_model_dict_lucie

In [21]:
# Compare questions: mmlu_pro_prompts_examples.json (from local JSONL) vs single_model_dict (from leaderboard pickle)
import string

def doc_to_query(doc):
    """Build query string from doc (question + options + Answer:)."""
    s = f"{doc['question']}\n"
    for i in range(len(doc["options"])):
        s += f"{string.ascii_uppercase[i]}. {doc['options'][i]}\n"
    s += "Answer:"
    return s

out_path = os.path.join(ROOT_PATH, "notebooks", "mmlu_pro_prompts_examples.json")
with open(out_path) as f:
    json_examples = json.load(f)

n_json = len(json_examples)
n_pickle = len(single_model_dict["doc"])
print(f"Count: JSON={n_json}, pickle={n_pickle}, match={n_json == n_pickle}")

if n_json != n_pickle:
    print("Length mismatch — cannot compare element-wise.")
else:
    query_matches = []
    full_prompt_matches = []
    for i in range(n_json):
        json_query = json_examples[i]["query"]
        json_full_prompt = json_examples[i]["full_prompt"]
        doc_i = single_model_dict["doc"][i]
        pickle_query = doc_to_query(doc_i)
        pickle_full_prompt = single_model_dict["arguments"][i]["gen_args_0"]["arg_0"]
        query_matches.append(json_query == pickle_query)
        full_prompt_matches.append(json_full_prompt == pickle_full_prompt)
    print(f"Query match: {sum(query_matches)}/{n_json}")
    print(f"Full prompt match: {sum(full_prompt_matches)}/{n_json}")
    if sum(query_matches) != n_json or sum(full_prompt_matches) != n_json:
        diff_idxs = [i for i in range(n_json) if not query_matches[i] or not full_prompt_matches[i]]
        print(f"First few differing indices: {diff_idxs[:10]}")

# Check whether queries match up to permutation (same set/multiset of queries)
from collections import Counter

json_queries = [ex["query"] for ex in json_examples]
pickle_queries = [doc_to_query(doc) for doc in single_model_dict["doc"]]

json_set = set(json_queries)
pickle_set = set(pickle_queries)
json_in_pickle = json_set <= pickle_set
pickle_in_json = pickle_set <= json_set
print(f"\n--- Match up to permutation (set of queries) ---")
print(f"Every JSON query appears in pickle: {json_in_pickle} (JSON set size {len(json_set)}, pickle set size {len(pickle_set)})")
print(f"Every pickle query appears in JSON: {pickle_in_json}")
if not json_in_pickle:
    only_in_json = json_set - pickle_set
    print(f"Queries in JSON but not in pickle: {len(only_in_json)} (first 3: {list(only_in_json)[:3]})")
if not pickle_in_json:
    only_in_pickle = pickle_set - json_set
    print(f"Queries in pickle but not in JSON: {len(only_in_pickle)} (first 3: {list(only_in_pickle)[:3]})")

json_counts = Counter(json_queries)
pickle_counts = Counter(pickle_queries)
multiset_match = json_counts == pickle_counts
print(f"Same multiset (identical counts per query): {multiset_match}")

Count: JSON=12032, pickle=12032, match=True
Query match: 2/12032
Full prompt match: 2/12032
First few differing indices: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

--- Match up to permutation (set of queries) ---
Every JSON query appears in pickle: False (JSON set size 12032, pickle set size 11874)
Every pickle query appears in JSON: False
Queries in JSON but not in pickle: 644 (first 3: ['25. Consider the initial value problem\n$$\n2 y^{\\prime \\prime}+3 y^{\\prime}-2 y=0, \\quad y(0)=1, \\quad y^{\\prime}(0)=-\\beta,\n$$\nwhere $\\beta>0$.\nFind the smallest value of $\\beta$ for which the solution has no minimum point.\nA. 7\nB. 2.5\nC. 4\nD. 1\nE. 2\nF. 6\nG. 5\nH. 1.5\nI. 0.5\nJ. 3\nAnswer:', 'A particle of mass $m$ and velocity $u_1$ makes a head-on collision with another particle of mass $2 m$ at rest. If the coefficient of restitution is such to make the loss of total kinetic energy a maximum, what are the velocities $v_1$ after the collision?\nA. $0$\nB. $\\frac{1}{3}$$u_1$\nC. $\\frac

In [9]:
# Compare single_model_dict_lucie vs single_model_dict: identical queries (and full_prompt structure: 5 examples + query)
import string
from collections import Counter

def doc_to_query(doc):
    """Build query string from doc (question + options + Answer:)."""
    s = f"{doc['question']}\n"
    for i in range(len(doc["options"])):
        s += f"{string.ascii_uppercase[i]}. {doc['options'][i]}\n"
    s += "Answer:"
    return s

lucie_queries = [doc_to_query(doc) for doc in single_model_dict_lucie["doc"]]
other_queries = [doc_to_query(doc) for doc in single_model_dict["doc"]]

n_lucie = len(lucie_queries)
n_other = len(other_queries)
print(f"Count: lucie={n_lucie}, single_model_dict={n_other}, match={n_lucie == n_other}")

# Set comparison: same set of queries?
lucie_set = set(lucie_queries)
other_set = set(other_queries)
lucie_in_other = lucie_set <= other_set
other_in_lucie = other_set <= lucie_set
print(f"\n--- Match up to permutation (set of queries) ---")
print(f"Every lucie query appears in single_model_dict: {lucie_in_other} (lucie set size {len(lucie_set)}, other set size {len(other_set)})")
print(f"Every single_model_dict query appears in lucie: {other_in_lucie}")
if not lucie_in_other:
    only_lucie = lucie_set - other_set
    print(f"Queries in lucie but not in single_model_dict: {len(only_lucie)} (first 3: {list(only_lucie)[:3]})")
if not other_in_lucie:
    only_other = other_set - lucie_set
    print(f"Queries in single_model_dict but not in lucie: {len(only_other)} (first 3: {list(only_other)[:3]})")

# Multiset: same counts per query?
lucie_counts = Counter(lucie_queries)
other_counts = Counter(other_queries)
multiset_match = lucie_counts == other_counts
print(f"Same multiset (identical counts per query): {multiset_match}")

# Structure check: full_prompt ends with the corresponding query (5 examples + query)
def full_prompt_ends_with_query(d, idx):
    full_prompt = d["arguments"][idx]["gen_args_0"]["arg_0"]
    query = doc_to_query(d["doc"][idx])
    return full_prompt.strip().endswith(query.strip()) or query.strip() in full_prompt[-len(query)-50:]

lucie_structure_ok = all(full_prompt_ends_with_query(single_model_dict_lucie, i) for i in range(n_lucie))
other_structure_ok = all(full_prompt_ends_with_query(single_model_dict, i) for i in range(n_other))
print(f"\n--- Full prompt structure (ends with query) ---")
print(f"Lucie: all full_prompts end with corresponding query: {lucie_structure_ok}")
print(f"single_model_dict: all full_prompts end with corresponding query: {other_structure_ok}")

Count: lucie=12032, single_model_dict=12032, match=True

--- Match up to permutation (set of queries) ---
Every lucie query appears in single_model_dict: False (lucie set size 11874, other set size 11873)
Every single_model_dict query appears in lucie: False
Queries in lucie but not in single_model_dict: 7 (first 3: ['Let {N(t), t=[0, \\infty]} be a Poisson process with rate $\\lambda = 0.5$. Find the probability of no arrivals in [3, 5)\nA. 0.82\nB. 0.99\nC. 0.50\nD. 0.75\nE. 0.25\nF. 0.01\nG. 0.67\nH. 0.37\nI. 0.60\nJ. 0.55\nAnswer:', 'Which of the following is a renewable energy source?\nA. Oil shale\nB. Wood\nC. Natural gas\nD. Oil\nE. Peat\nF. Diesel\nG. Gasoline\nH. Propane\nI. Coal\nJ. Uranium\nAnswer:', 'Suppose there are 100 identical firms in a perfectly competitive industry. Each firm has a short-run total cost function of the form C(q) = \\frac{1}{300}q^3 + 0.2q^2 + 4q + 10. Suppose market demand is given by Q = -200P + 8,000. What will be the short-run equilibrium price?\n